# SI7011 · Sesión 06 · Entrenar el *checkpoint* largo del GPT (solo profesor)

Produce `gpt_quijote.pt`, el archivo del modelo de Kaggle `juanmartinezv4399/si7011-s06-gpt-quijote` que carga la sección 6 del notebook 6.3. El texto, el tokenizador, el modelo y el loop son **copias exactas** de las celdas del notebook 6.3; lo único que cambia es el número de pasos.

**Cómo correrlo en Kaggle**
1. *Settings*: GPU (T4 o P100) e **Internet** activado (para bajar el texto de Gutenberg), o *Add Input* con un dataset que tenga `pg2000.txt`.
2. **Save Version → Save & Run All (Commit)**.
3. Al terminar, descarga `gpt_quijote.pt` de *Output* y súbelo como modelo `si7011-s06-gpt-quijote` (framework PyTorch, variación `default`), público.

**Si no alcanza el tiempo.** Guarda `gpt_quijote_parcial.pt` cada `GUARDAR_CADA` pasos y se detiene antes de `MAX_HORAS`. Para seguir: versión nueva, *Add Input* → *Your Work* → la salida anterior, y otra vez *Save & Run All*.

In [ ]:
PASOS_LARGO = 30000           # el notebook 6.3 lee este número del checkpoint
GUARDAR_CADA = 2000
MAX_HORAS = 11.0
INICIO = __import__('time').time()

In [ ]:
import json, math, os, re, time, urllib.request
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F

try:
    from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers
except ImportError:                                        # Kaggle y Colab ya la traen
    os.system('pip install -q tokenizers')
    from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers

torch.manual_seed(0)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '·', dev)

In [ ]:
GUTENBERG = 'https://www.gutenberg.org/cache/epub/2000/pg2000.txt'    # Don Quijote, dominio público
ROOTS = [Path('.'), Path('data'), Path('/kaggle/input'), Path('/teamspace/studios/this_studio/data')]

def buscar(*patrones):
    for d in ROOTS:
        for patron in patrones:
            hits = sorted(d.glob(f'**/{patron}')) if d.exists() else []
            if hits:
                return hits[0]
    return None

def cargar_texto():
    ruta = buscar('pg2000.txt', 'quijote*.txt')
    if ruta is not None:
        return ruta.read_text(encoding='utf-8'), f'archivo local ({ruta})'
    Path('data').mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(GUTENBERG, 'data/pg2000.txt')
        return Path('data/pg2000.txt').read_text(encoding='utf-8'), 'descargado de Project Gutenberg'
    except Exception as e:
        raise RuntimeError('Sin internet: pon pg2000.txt (Don Quijote, Project Gutenberg) en data/. '
                           'En Kaggle, activa Internet en Settings.') from e

crudo, fuente = cargar_texto()
inicio, fin = crudo.find('*** START OF'), crudo.find('*** END OF')
texto = crudo[crudo.find('\n', inicio) + 1:fin] if inicio >= 0 and fin > 0 else crudo   # sin la licencia de Gutenberg
texto = re.sub(r'\r\n?', '\n', texto).strip()
corte = int(0.9 * len(texto))
texto_train, texto_val = texto[:corte], texto[corte:]                # los últimos capítulos quedan para validar
print(f'fuente: {fuente} · {len(texto):,} caracteres · {len(set(texto))} caracteres distintos')
FRASE = 'En un lugar de la Mancha, de cuyo nombre no quiero acordarme'          # la primera frase del libro

In [ ]:
def entrenar_bpe(n_vocab):
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)   # parte en bytes: nunca hay [UNK]
    tok.decoder = decoders.ByteLevel()
    tr = trainers.BpeTrainer(vocab_size=n_vocab, min_frequency=2, show_progress=False,
                             initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
    tok.train_from_iterator([texto_train], tr)
    return tok

In [ ]:
N_VOCAB = 4000
tok = tokenizadores[N_VOCAB] if 'tokenizadores' in globals() else entrenar_bpe(N_VOCAB)
ids_train = torch.tensor(tok.encode(texto_train).ids, dtype=torch.long)
ids_val = torch.tensor(tok.encode(texto_val).ids, dtype=torch.long)
print(f'{len(ids_train):,} tokens de entrenamiento · {len(ids_val):,} de validación')

def lote(ids, B, T):
    """B ventanas de T + 1 tokens tomadas al azar: la entrada son los primeros T, el objetivo los últimos T."""
    i = torch.randint(len(ids) - T - 1, (B,))
    x = torch.stack([ids[j:j + T] for j in i])
    y = torch.stack([ids[j + 1:j + T + 1] for j in i])               # el mismo texto, corrido un lugar
    return x.to(dev), y.to(dev)

In [ ]:
class Bloque(nn.Module):
    """El bloque pre-norm del notebook 6.2, solo con autoatención causal."""
    def __init__(self, d, H, p=0.1):
        super().__init__()
        self.H = H
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)                                # W_Q, W_K, W_V juntas
        self.W_O = nn.Linear(d, d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.drop, self.p = nn.Dropout(p), p

    def atencion(self, X):
        B, T, d = X.shape
        Q, K, V = self.qkv(X).view(B, T, 3, self.H, d // self.H).permute(2, 0, 3, 1, 4)   # (B, H, T, d_k) cada una
        Y = F.scaled_dot_product_attention(Q, K, V, is_causal=True,                       # máscara causal
                                           dropout_p=self.p if self.training else 0.0)
        return self.W_O(Y.transpose(1, 2).reshape(B, T, d))

    def forward(self, X):
        X = X + self.drop(self.atencion(self.ln1(X)))                # X ← X + MHA(LN(X))
        return X + self.drop(self.mlp(self.ln2(X)))                  # X ← X + MLP(LN(X))

class GPT(nn.Module):
    def __init__(self, n_vocab=N_VOCAB, d=192, H=6, D=4, T_max=256):
        super().__init__()
        self.T_max = T_max
        self.E = nn.Embedding(n_vocab, d)                             # embedding de tokens
        self.P = nn.Embedding(T_max, d)                               # p_i aprendidos
        self.bloques = nn.ModuleList([Bloque(d, H) for _ in range(D)])
        self.ln = nn.LayerNorm(d)
        self.W_U = nn.Linear(d, n_vocab, bias=False)
        self.W_U.weight = self.E.weight                               # atadas: la misma matriz (diapositiva 29)
        self.apply(self._iniciar)

    @staticmethod
    def _iniciar(m):                                                  # la inicialización de GPT-2: N(0, 0.02²)
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, ids):                                           # ids: (B, T)
        X = self.E(ids) + self.P(torch.arange(ids.shape[1], device=ids.device))
        for b in self.bloques:
            X = b(X)
        return self.W_U(self.ln(X))                                   # logits (B, T, |V|)

    @torch.no_grad()
    def generar(self, prompt, n=60, tau=1.0, top_k=None, greedy=False, seed=0):
        g = torch.Generator(device='cpu').manual_seed(seed)
        ids = torch.tensor([tok.encode(prompt).ids], device=dev)
        for _ in range(n):
            z = self(ids[:, -self.T_max:])[0, -1]                     # logits de la ÚLTIMA posición
            if greedy:
                sig = z.argmax()
            else:
                z = z / tau                                           # temperatura
                if top_k:
                    corte = z.topk(top_k).values[-1]
                    z = z.masked_fill(z < corte, float('-inf'))       # solo los k más probables
                sig = torch.multinomial(z.softmax(-1).cpu(), 1, generator=g)[0].to(dev)
            ids = torch.cat([ids, sig.view(1, 1)], 1)                 # el token producido es la siguiente entrada
        return tok.decode(ids[0].tolist())

In [ ]:
@torch.no_grad()
def perdida_val(modelo, n=20, B=32, T=128):
    modelo.eval()
    torch.manual_seed(123)                                            # las mismas ventanas cada vez
    L = np.mean([F.cross_entropy(modelo(x).flatten(0, 1), y.flatten()).item()
                 for x, y in (lote(ids_val, B, T) for _ in range(n))])
    modelo.train()
    return L

def entrenar(modelo, pasos, B=64, T=128, lr=1e-3, warmup=100, cada=100, muestras_en=(), hist=None, paso0=0,
             opt=None, total=None):
    """J = la entropía cruzada media sobre las B·T posiciones: T predicciones por ventana, en una pasada."""
    total = total or pasos
    opt = opt or torch.optim.AdamW(modelo.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    f = lambda s: (s + 1) / warmup if s < warmup else 0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, total - warmup)))
    hist = hist if hist is not None else {'paso': [], 'train': [], 'val': [], 'muestras': {}}
    modelo.train()
    t0, acum = time.time(), []
    for s in range(paso0, paso0 + pasos):
        for gr in opt.param_groups:
            gr['lr'] = lr * f(s)                                      # warmup y coseno (sesión 02)
        x, y = lote(ids_train, B, T)
        loss = F.cross_entropy(modelo(x).flatten(0, 1), y.flatten())
        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
        opt.step()
        acum.append(loss.item())
        if (s + 1) % cada == 0:
            hist['paso'].append(s + 1); hist['train'].append(float(np.mean(acum))); acum = []
            hist['val'].append(float(perdida_val(modelo)))
            print(f'paso {s + 1:6d} · train {hist["train"][-1]:.3f} · val {hist["val"][-1]:.3f} '
                  f'· {time.time() - t0:.0f} s')
        if (s + 1) in muestras_en:
            modelo.eval()
            hist['muestras'][s + 1] = modelo.generar('En un lugar de la Mancha', n=40, tau=0.8, seed=1)
            modelo.train()
            print('   muestra:', repr(hist['muestras'][s + 1]))
    return hist, opt

In [ ]:
import shutil
CONFIG = dict(n_vocab=N_VOCAB, d=192, H=6, D=4, T_max=256)
final, parcial = Path('gpt_quijote.pt'), Path('gpt_quijote_parcial.pt')
previo = buscar('gpt_quijote_parcial.pt')
torch.manual_seed(0)
modelo = GPT(**CONFIG).to(dev)
opt = torch.optim.AdamW(modelo.parameters(), lr=1e-3, betas=(0.9, 0.95), weight_decay=0.1)
hist, hecho = None, 0
if previo is not None:
    ck = torch.load(previo, map_location=dev, weights_only=False)
    tok = Tokenizer.from_str(ck['tokenizador'])
    modelo.load_state_dict(ck['modelo']); opt.load_state_dict(ck['opt'])
    hist, hecho = ck['historia'], ck['pasos']
    print('retomo en el paso', hecho)

def guardar(ruta, pasos):
    torch.save({'modelo': modelo.state_dict(), 'opt': opt.state_dict(), 'config': CONFIG, 'pasos': pasos,
                'historia': hist, 'tokenizador': tok.to_str()}, ruta)

while hecho < PASOS_LARGO:
    tramo = min(GUARDAR_CADA, PASOS_LARGO - hecho)
    hist, opt = entrenar(modelo, tramo, cada=500, hist=hist, paso0=hecho, opt=opt, total=PASOS_LARGO,
                         muestras_en=(hecho + tramo,))
    hecho += tramo
    guardar(parcial, hecho)
    if (__import__('time').time() - INICIO) / 3600 > MAX_HORAS and hecho < PASOS_LARGO:
        print('Me detengo por tiempo en el paso', hecho, '· retoma en una versión nueva'); break
if hecho >= PASOS_LARGO:
    ck = torch.load(parcial, map_location=dev, weights_only=False)
    ck.pop('opt')                                       # el notebook 6.3 no lo necesita
    torch.save(ck, final)
    print('listo:', final, f'· val {hist["val"][-1]:.3f}')